# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and analyzing the FAIR² mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. The Croissant schema describes all available data files and fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset (metadata and access records)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title:", metadata.name)
print("Description:", metadata.description)
print("Published:", getattr(metadata, 'datePublished', ''))
print("Version:", getattr(metadata, 'version', ''))

## 2. Data Overview
Let's list all available record sets and their contained fields. All entities are referenced by their `@id` field. This will help us understand the dataset structure and select relevant components for analysis.

In [ ]:
# List available record sets and their field @ids
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
elif hasattr(metadata, 'recordSet'):
    record_sets = metadata.recordSet
else:
    record_sets = []

# Print high-level information about record sets and their fields (@id)
record_set_ids = []
field_lookup = {}

for rs in record_sets:
    print(f"Record set name: {getattr(rs, 'name', '')}")
    print(f"  @id: {getattr(rs, '@id', '')}")
    record_set_ids.append(getattr(rs, '@id', ''))
    if hasattr(rs, 'fields'):
        fields = rs.fields
    elif hasattr(rs, 'field'):
        fields = rs.field if isinstance(rs.field, list) else [rs.field]
    else:
        fields = []
    print("  Fields and columns in this record set:")
    for f in fields:
        # Each field may have a name, @id, and a reference to a column
        print(f"    - {getattr(f, 'name', '')} [@id: {getattr(f, '@id', '')}]")
        if hasattr(f, 'column'):
            cols = f.column if isinstance(f.column, list) else [f.column]
            for col in cols:
                print(f"        Column: {getattr(col, 'name', '')} [@id: {getattr(col, '@id', '')}]")
        field_lookup[getattr(f, 'name', '')] = getattr(f, '@id', '')
    print()
if not record_set_ids:
    print("No record sets discovered in the Croissant schema.")

## 3. Data Extraction
Load all records from a chosen record set into a DataFrame for downstream processing. For reproducibility, always reference with the explicit record set `@id`.

In [ ]:
# For this dataset, let's attempt to infer the main record set @id.
# As of schema version 1.0, many protein tables are found as a single record set.

# If no record set found above, try to list them programmatically:
rs_ids = [r for r in record_set_ids if r]  # List of non-empty @id
if not rs_ids:
    # Try to find @id from dataset metadata, fallback for single main record set
    if hasattr(metadata, 'recordSet') and len(metadata.recordSet) == 0:
        print("Warning: No explicit recordSet listed in this object's Croissant metadata. You may need to check the schema file or documentation for the primary record set @id.")
    # Use the following default (as typical with Croissant mass spec datasets):
    main_record_set_id = 'cr:recordSet_1'
else:
    main_record_set_id = rs_ids[0]  # If multiple, select the first for exploration

# Extract records for the main record set
try:
    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} records from record set {main_record_set_id}.")
    print("Columns:", df.columns.tolist())
    display(df.head())
except Exception as e:
    print(f"Cannot load records for record set {main_record_set_id} due to: {e}")

## 4. Exploratory Data Analysis (EDA)
You can process and analyze numerical or categorical fields. In this section we select a numeric field (e.g., `MW` for molecular weight or a protein abundance), filter records by value, normalize, and group entries by another field like `description` or another attribute (referenced by their `@id`).

In [ ]:
# Typical protein tables have columns like 'MW', 'coverage', 'peptide_count', 'abundance', etc.
# Let's try to select 'MW' (molecular weight) for numeric analysis.
# Make sure to reference by the exact DataFrame field name seen above.

if 'MW' in df.columns:
    numeric_field = 'MW'  # Example: Molecular Weight (kDa)
elif 'mw' in df.columns:
    numeric_field = 'mw'
elif len(df.select_dtypes(include='number').columns) > 0:
    numeric_field = df.select_dtypes(include='number').columns[0]
else:
    numeric_field = df.columns[0]

threshold = 50000  # For MW, this could be 50,000 Da
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered for {numeric_field} > {threshold} ({len(filtered_df)} records):")
display(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} in the filtered subset:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by another field, e.g., 'description' if present
if 'description' in df.columns:
    group_field = 'description'
elif 'accession' in df.columns:
    group_field = 'accession'  # often unique
else:
    group_field = df.columns[0]

if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by '{group_field}':")
    display(grouped_df.head())

## 5. Visualization
Below we plot the distribution of the selected numeric field (e.g., MW) and relationships between numeric fields if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field in the main df
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), bins=40, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If there's another numeric field, plot scatter
num_fields = df.select_dtypes(include='number').columns.tolist()
if len(num_fields) > 1:
    other_numeric = [f for f in num_fields if f != numeric_field]
    plt.figure(figsize=(7,5))
    sns.scatterplot(x=df[numeric_field], y=df[other_numeric[0]])
    plt.xlabel(numeric_field)
    plt.ylabel(other_numeric[0])
    plt.title(f"{numeric_field} vs {other_numeric[0]}")
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² mass spectrometry dataset defined by Croissant, examined its fields and structure by `@id`, and performed basic data analysis and visualization on protein features such as molecular weight. Further biological or technical investigation can be performed as required for your research questions.